# V7.1 — matched dataset and WRITTEN-in-both gate

This notebook uses the frozen reciprocal roster, group split, L16 J-Lens directions, and calibration threshold. Engine and dashboard prompts share the exact explicit-concept prefix and use the identical `logit(answer_A) - logit(answer_B)` metric. The dashboard alone states the answer and asks the model to copy it.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PATH'] = '/home/jovyan/.local/bin:/home/jovyan/.npm-global/bin:' + os.environ['PATH']
os.environ['HF_HOME'] = '/home/jovyan/.cache/huggingface'
os.environ['HF_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/jovyan/.cache/huggingface/hub'

for command in [
    ['/usr/bin/which', 'hf'],
    ['/home/jovyan/.local/bin/hf', 'auth', 'whoami'],
    ['nvidia-smi', '--query-gpu=memory.total,memory.free', '--format=csv'],
]:
    print(subprocess.check_output(command, text=True).strip())

/home/jovyan/.local/bin/hf


user=sushmanth orgs=sushmanthreddy,devoworm-group,OWG,syscv-community,context-course
memory.total [MiB], memory.free [MiB]
143771 MiB, 143072 MiB


In [2]:
from src.data_gen_v7 import run_dataset_stage_v7

def progress(event, payload):
    print(f'{event}: {dict(payload)}', flush=True)

summary = run_dataset_stage_v7(progress=progress)
summary

load_model: {'model': 'Qwen/Qwen2.5-7B-Instruct', 'revision': 'a09a35458c702b33eeacc393d103063234e8bc28'}


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

build_candidates: {'minimum': 90}


hf_jlens_kl: {'n_prompts': 20}


/home/jovyan/j-space-thoughts/.venv/lib/python3.11/site-packages/transformers/models/qwen2/modeling_qwen2.py:108: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)


...

directions: {'layer': 16, 'n_tokens': 66}


verify: {'candidate_pairs': 118}


/home/jovyan/j-space-thoughts/src/jlens_interface.py:401: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  directions = F.normalize(rows @ jacobian, dim=-1)


complete: {'candidates': 118, 'calibration_pairs': 25, 'calibration_gate_pass': 24, 'evaluation_pairs': 93, 'verified_pairs': 77, 'unverified_pairs': 16, 'manifest_sha256': '536fac395f693949b4f6d3d2961c1deac07e02669011cc567b641eee92181f08', 'dataset_path': '/home/jovyan/j-space-thoughts/results/v7/raw/matched_dataset_v7.json'}


{'dataset_path': '/home/jovyan/j-space-thoughts/results/v7/raw/matched_dataset_v7.json',
 'manifest_path': '/home/jovyan/j-space-thoughts/results/v7/matched_manifest_v7.json',
 'manifest_sha256': '536fac395f693949b4f6d3d2961c1deac07e02669011cc567b641eee92181f08',
 'counts': {'candidates': 118,
  'calibration_pairs': 25,
  'calibration_gate_pass': 24,
  'evaluation_pairs': 93,
  'verified_pairs': 77,
  'unverified_pairs': 16},
 'logit_agreement': {'n': 20,
  'threshold': 0.001,
  'max_mean_kl': 1.6602172081547906e-08},
 'direction_digest_sha256': 'ec43eef29e4fcd9f3063059c6d5d2e87e56ef56dc16d9305f8bd38988a1f758a',
 'target_verified_pairs_met': True}

In [3]:
import json
from collections import Counter

dataset = json.loads(Path(summary['dataset_path']).read_text())
manifest = json.loads(Path(summary['manifest_path']).read_text())
verified = [row for row in manifest['rows'] if row['verification_status'] == 'VERIFIED']
unverified = [row for row in manifest['rows'] if row['verification_status'] == 'UNVERIFIED']
assert dataset['target_verified_pairs_met']
assert manifest['metric_contract']['engine'] == manifest['metric_contract']['dashboard']
assert all(row['engine_metric'] == row['dashboard_metric'] for row in manifest['rows'])
assert all(row['same_metric_token_ids'] and row['same_answer_type'] for row in manifest['rows'])
assert all('2 + 2' not in row['engine_prompt_a'] + row['dashboard_prompt_a'] for row in manifest['rows'])
assert all(row['engine_position_a'] == row['dashboard_position_a'] for row in manifest['rows'])
assert all(row['engine_position_b'] == row['dashboard_position_b'] for row in manifest['rows'])

audit = {
    'counts': manifest['counts'],
    'kl': manifest['logit_agreement'],
    'verification_reasons': Counter(reason for row in unverified for reason in row['verification_reasons']),
    'max_engine_dashboard_written_delta': max(
        max(abs(row['engine_z_a'] - row['dashboard_z_a']), abs(row['engine_z_b'] - row['dashboard_z_b']))
        for row in manifest['rows']
    ),
    'example': {
        key: verified[0][key]
        for key in [
            'pair_id', 'engine_prompt_a', 'dashboard_prompt_a',
            'answer_a_token_id', 'answer_b_token_id',
            'engine_metric', 'dashboard_metric',
            'engine_z_a', 'dashboard_z_a',
        ]
    },
}
audit

{'counts': {'calibration_gate_pass': 24,
  'calibration_pairs': 25,
  'candidates': 118,
  'evaluation_pairs': 93,
  'unverified_pairs': 16,
  'verified_pairs': 77},
 'kl': {'max_mean_kl': 1.6602172081547906e-08,
  'n': 20,
  'status': 'PASS',
  'threshold': 0.001},
 'verification_reasons': Counter({'ENGINE_B_CONCEPT_NOT_WRITTEN': 9,
          'DASHBOARD_B_CONCEPT_NOT_WRITTEN': 9,
          'ENGINE_A_CONCEPT_NOT_WRITTEN': 8,
          'DASHBOARD_A_CONCEPT_NOT_WRITTEN': 8}),
 'max_engine_dashboard_written_delta': 0.0,
 'example': {'pair_id': 'symmetric-000',
  'engine_prompt_a': 'Fact: the element with atomic number 13 is aluminum. Its chemical symbol is',
  'dashboard_prompt_a': 'Fact: the element with atomic number 13 is aluminum. Its chemical symbol is Al. Copy the chemical symbol exactly:',
  'answer_a_token_id': 1674,
  'answer_b_token_id': 72593,
  'engine_metric': 'logit(answer_A) - logit(answer_B)',
  'dashboard_metric': 'logit(answer_A) - logit(answer_B)',
  'engine_z_a': 3.829